## Configuration

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
import os
# Path configuration
# path_file = '/content/drive/MyDrive/tesisugm/'  # google colab
path_file = ''  # local
file_path = os.path.join(path_file, "", "hybrid_oversampling.py")

In [3]:
%%writefile $file_path

# Nama File: hybrid_oversampling.py
from __future__ import annotations
from collections import Counter
import numpy as np
import math
import time
import pandas as pd
from scipy.sparse import issparse, csr_matrix, vstack, diags
from sklearn.base import BaseEstimator
from sklearn.utils import check_random_state
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN, MiniBatchKMeans

_EPS_NUMERICAL = 1e-9
_SYNTH_DTYPE   = np.uint8

# ==============================================================
# 1. ASTRA-SMOTE (Adaptive Safe-Threshold Resampling Algorithm SMOTE)
# ==============================================================
class ASTRA_SMOTE(BaseEstimator):
    """
    Adaptive Safe-Threshold Resampling Algorithm SMOTE (ASTRA-SMOTE)

    Mekanisme:
    1. MiniBatchKMeans clustering di seluruh training data
    2. Identifikasi safe zone: cluster dengan purity >= safe_threshold
    3. Generate sintetis HANYA dari safe zone
    4. Jumlah yang di-generate proporsional terhadap safe_ratio (alpha adaptif)

    Parameter tuning:
    - n_clusters      : jumlah cluster KMeans
    - safe_threshold  : minimum purity cluster untuk masuk safe zone
    - alpha_mode      : 'adaptive' (default) atau 'full'
    """
    def __init__(self,
                 n_clusters=30,
                 k_neighbors=5,
                 metric='cosine',
                 nn_algorithm='brute',
                 k_neighbors_min=2,
                 safe_threshold=0.30,
                 alpha_mode='adaptive',      # ← kontrol effective_target
                 sampling_strategy=None,
                 batch_size=5000,
                 kmeans_batch_size=2048,
                 kmeans_n_init=10,
                 min_yield_ratio=0.4,
                 max_bad_batches=2,
                 max_iters=30,
                 normalize_syn=False,
                 verbose=1,
                 log_every=5,
                 random_state=42,
                 nn_rebuild_every=5):

        self.n_clusters        = int(n_clusters)
        self.k_neighbors       = int(k_neighbors)
        self.metric            = metric
        self.nn_algorithm      = nn_algorithm
        self.k_neighbors_min   = int(k_neighbors_min)
        self.safe_threshold    = float(safe_threshold)
        self.alpha_mode        = alpha_mode
        self.sampling_strategy = sampling_strategy or {}
        self.batch_size        = int(batch_size)
        self.kmeans_batch_size = int(kmeans_batch_size)
        self.kmeans_n_init     = int(kmeans_n_init)
        self.min_yield_ratio   = float(min_yield_ratio)
        self.max_bad_batches   = int(max_bad_batches)
        self.max_iters         = int(max_iters)
        self.normalize_syn     = bool(normalize_syn)
        self.verbose           = int(verbose)
        self.log_every         = int(max(1, log_every))
        self.random_state      = random_state
        self.nn_rebuild_every  = int(max(1, nn_rebuild_every))
        self.label_names_      = None
        self.metadata_         = {}
        self.history_          = []
        self.is_fitted_        = False
        self.n_original_       = None

    def _ts(self): return time.strftime("%H:%M:%S")

    def _log(self, level, msg):
        if self.verbose >= level:
            print(f"[{self._ts()}] {msg}")
        self.history_.append((self._ts(), level, msg))

    def _rng(self):
        return check_random_state(self.random_state)

    def _row_scale(self, X_csr, weights):
        X_csr = X_csr.tocsr()
        w     = np.asarray(weights, dtype=np.float32).ravel()
        assert w.shape[0] == X_csr.shape[0], \
            f"weights {w.shape[0]} != rows {X_csr.shape[0]}"
        D = diags(w, offsets=0, shape=(len(w), len(w)), format='csr')
        return D.dot(X_csr)

    def _gen_synth_batch(self, X_safe, nn, b, rng):
        n = X_safe.shape[0]
        if n < 2 or b <= 0:
            return None
        i_idx        = rng.randint(0, n, size=b)
        _, neighbors = nn.kneighbors(X_safe[i_idx])
        k_avail      = neighbors.shape[1] - 1
        if k_avail <= 0:
            return None
        pick  = rng.randint(0, k_avail, size=b)
        j_idx = neighbors[np.arange(b), 1 + pick]
        alpha = rng.rand(b).astype(np.float32)
        Xi    = self._row_scale(X_safe[i_idx], 1.0 - alpha)
        Xj    = self._row_scale(X_safe[j_idx], alpha)
        X_syn = (Xi + Xj).tocsr()
        if self.normalize_syn:
            norm = np.sqrt(X_syn.multiply(X_syn).sum(axis=1)).A1
            nz   = norm > 0
            if nz.any():
                X_syn[nz] = X_syn[nz].multiply((1.0 / norm[nz])[:, None])
        return X_syn

    def fit_resample(self, X, y, label_names=None):
        t0  = time.time()
        rng = self._rng()

        if not issparse(X):
            X = csr_matrix(X)
        X = X.astype(np.float32).tocsr()
        y = np.asarray(y)

        self.label_names_ = label_names
        self.n_original_  = X.shape[0]

        counts    = Counter(y)
        maj_class = max(counts, key=counts.get)

        self._log(1, f"📋 Distribusi awal   : {dict(counts)}")
        self._log(1, f"🏷️  Mayoritas         : {maj_class} ({counts[maj_class]})")
        self._log(1, f"🎯 Sampling strategy : {self.sampling_strategy}")
        self._log(1, f"🔧 alpha_mode        : {self.alpha_mode}")

        # ── Clustering ────────────────────────────────────────
        t_cl = time.time()
        self._log(1, f"📌 MiniBatchKMeans n_clusters={self.n_clusters}...")
        km = MiniBatchKMeans(
            n_clusters  = self.n_clusters,
            random_state= self.random_state,   # ← fix: tidak hardcode 0
            n_init      = self.kmeans_n_init,
            batch_size  = self.kmeans_batch_size,
        )
        clusters = km.fit_predict(X)
        cl_idx   = [np.where(clusters == c)[0] for c in range(self.n_clusters)]
        self._log(1, f"✅ Clustering: {time.time()-t_cl:.2f}s")

        X_res, y_res = X, y
        total_syn    = 0
        class_meta   = {}
        min_cls_size = min(counts.values())
        k_safe_global = max(self.k_neighbors_min,
                            min(self.k_neighbors + 1, min_cls_size))
        is_cls_cache  = {c: (y == c) for c in counts}

        for cls, cur_n in counts.items():
            cls_int = int(cls)
            cname   = (label_names[cls_int]
                       if label_names is not None and cls_int < len(label_names)
                       else f"class_{cls}")

            if cls_int not in self.sampling_strategy:
                strategy = "skip_majority" if cls == maj_class \
                           else "skip_not_in_strategy"
                class_meta[cls] = {
                    "strategy": strategy, "original": cur_n,
                    "generated": 0, "final": cur_n,
                }
                self._log(1, f"⏭️  {cname}: {strategy}")
                continue

            target_n = int(self.sampling_strategy[cls_int])
            if target_n <= cur_n:
                class_meta[cls] = {
                    "strategy": "skip_already_enough", "original": cur_n,
                    "generated": 0, "final": cur_n, "target": target_n,
                }
                self._log(1, f"⏭️  {cname}: already enough ({cur_n} >= {target_n})")
                continue

            t_cls = time.time()

            # ── Safe set ──────────────────────────────────────
            is_cls   = is_cls_cache[cls]
            safe_idx = []
            for c_id, idxs in enumerate(cl_idx):
                if idxs.size == 0:
                    continue
                if is_cls[idxs].mean() >= self.safe_threshold:
                    safe_idx.extend(idxs[is_cls[idxs]].tolist())

            safe_n     = len(safe_idx)
            safe_ratio = safe_n / cur_n if cur_n > 0 else 0.0

            # Alpha adaptif
            thr   = self.safe_threshold
            alpha = (0.0 if safe_ratio <= thr else
                     min(1.0, (safe_ratio - thr) / max(_EPS_NUMERICAL, 1.0 - thr)))

            # Effective target
            if self.alpha_mode == 'adaptive':
                effective_target = int(cur_n + alpha * max(0, target_n - cur_n))
            else:  # 'full'
                effective_target = target_n

            self._log(1, f"\n🔍 {cname}: cur={cur_n} target={target_n} "
                         f"effective={effective_target} "
                         f"safe={safe_n}({safe_ratio:.1%}) α={alpha:.2f}")

            if safe_n < 2 or effective_target <= cur_n:
                reason = "skip_no_safe" if safe_n < 2 else "skip_alpha_zero"
                class_meta[cls] = {
                    "strategy"        : reason,
                    "original"        : cur_n, "generated": 0,
                    "final"           : cur_n, "safe_n": safe_n,
                    "target"          : target_n,
                    "effective_target": effective_target,
                }
                self._log(1, f"⚠️ {cname}: skip ({reason})")
                continue

            need   = effective_target - cur_n
            X_safe = X_res[safe_idx]
            k_use  = min(k_safe_global, X_safe.shape[0])
            nn     = NearestNeighbors(
                         n_neighbors=max(self.k_neighbors_min, k_use),
                         metric=self.metric,
                         algorithm=self.nn_algorithm,
                     ).fit(X_safe)

            blocks = []
            iters  = 0
            bad    = 0

            while sum(b.shape[0] for b in blocks) < need \
                  and iters < self.max_iters:
                remaining = need - sum(b.shape[0] for b in blocks)
                b_size    = min(self.batch_size, remaining)
                X_syn     = self._gen_synth_batch(X_safe, nn, b_size, rng)

                if X_syn is None or \
                   X_syn.shape[0] < int(self.min_yield_ratio * b_size):
                    bad += 1
                    if bad >= self.max_bad_batches:
                        self._log(1, f"⛔ {cname}: early stop (low-yield)")
                        break
                else:
                    bad = 0

                if X_syn is not None:
                    blocks.append(X_syn)
                    X_safe = vstack([X_safe, X_syn], format='csr')

                iters += 1
                if iters % self.nn_rebuild_every == 0:
                    k_use = min(k_safe_global, X_safe.shape[0])
                    nn    = NearestNeighbors(
                                n_neighbors=max(self.k_neighbors_min, k_use),
                                metric=self.metric,
                                algorithm=self.nn_algorithm,
                            ).fit(X_safe)

                if self.verbose >= 2 and iters % self.log_every == 0:
                    self._log(2, f"  iter={iters} "
                                 f"gen={sum(b.shape[0] for b in blocks)}")

            if blocks:
                X_syn_all = vstack(blocks, format='csr')
                y_syn     = np.full(X_syn_all.shape[0], cls, dtype=y_res.dtype)
                X_res     = vstack([X_res, X_syn_all], format='csr')
                y_res     = np.concatenate([y_res, y_syn])
                generated = X_syn_all.shape[0]
                total_syn += generated
            else:
                generated = 0

            class_meta[cls] = {
                "strategy"        : "astra_smote_adaptive_safe",
                "original"        : cur_n,
                "target"          : target_n,
                "effective_target": effective_target,
                "safe_n"          : safe_n,
                "safe_ratio"      : round(safe_ratio, 4),
                "alpha"           : round(alpha, 4),
                "generated"       : generated,
                "final"           : cur_n + generated,
                "iters"           : iters,
                "elapsed"         : round(time.time() - t_cls, 2),
            }
            self._log(1, f"🏁 {cname}: gen={generated} "
                         f"final={cur_n+generated} "
                         f"t={class_meta[cls]['elapsed']}s")

        self.metadata_ = {
            "summary": {
                "n_original"  : self.n_original_,
                "n_final"     : X_res.shape[0],
                "total_synth" : total_syn,
                "elapsed_sec" : round(time.time() - t0, 2),
            },
            "classes": class_meta,
        }
        self.is_fitted_ = True
        self._log(1, f"\n📦 Done: total_synth={total_syn} "
                     f"n_final={X_res.shape[0]} "
                     f"t={self.metadata_['summary']['elapsed_sec']}s")
        return X_res, y_res

    def print_summary(self):
        if not self.is_fitted_:
            raise ValueError("Belum di-fit.")
        s = self.metadata_["summary"]
        print(f"\n📊 ASTRA-SMOTE Summary")
        print(f"  Original : {s['n_original']:,}")
        print(f"  Final    : {s['n_final']:,}")
        print(f"  Synthetic: {s['total_synth']:,}")
        print(f"  Time     : {s['elapsed_sec']}s")
        for cls, v in self.metadata_["classes"].items():
            if v["strategy"] in ("skip_majority", "skip_not_in_strategy"):
                continue
            cname = (self.label_names_[int(cls)]
                     if self.label_names_ is not None
                     else f"class_{cls}")
            print(f"  {cname:12s}: {v['strategy']:25s} "
                  f"orig={v['original']:,} gen={v['generated']:,} "
                  f"final={v['final']:,} "
                  f"safe={v.get('safe_n',0):,}({v.get('safe_ratio',0):.1%}) "
                  f"α={v.get('alpha',0):.2f}")


# ==============================================================
# 2. SMOTE-RADIANT (Robust Auto-Density Inter-class Anomaly Neutralization Technique)
# ==============================================================
class SMOTE_RADIANT(BaseEstimator):
    """
    SMOTE-RADIANT (Robust Auto-Density Inter-class Anomaly Neutralization Technique)

    Mekanisme:
    1. SMOTE — generate sampel sintetis ke target per kelas
    2. DBSCAN — identifikasi dan buang noise (anomaly neutralization) dari hasil SMOTE
    3. Top-up (opsional) — jika strict_target=True, generate ulang
       sampel yang terbuang agar mencapai target

    Parameter tuning:
    - eps, min_samples    : parameter DBSCAN (atau 'auto')
    - clean_only_synth    : True = hanya buang sintetis yang noise
    - strict_target       : True = top-up SMOTE #2 setelah cleaning
    """
    def __init__(self,
                 sampling_strategy=None,
                 k_neighbors=5,
                 eps='auto',
                 min_samples='auto',
                 metric='cosine',
                 nn_algorithm='brute',
                 clean_mode='minor_only',
                 clean_only_synth=True,
                 strict_target=False,
                 eps_percentile=60,
                 eps_k_min=2,
                 eps_k_max=10,
                 eps_min_clip=1e-6,
                 eps_max_clip=1.0,
                 min_cluster_size=3,
                 batch_size=2000,
                 max_iters=100,
                 normalize_syn=False,
                 verbose=1,
                 random_state=42):

        self.sampling_strategy = sampling_strategy or {}
        self.k_neighbors       = int(k_neighbors)
        self.eps               = eps
        self.min_samples       = min_samples
        self.metric            = metric
        self.nn_algorithm      = nn_algorithm
        self.clean_mode        = clean_mode
        self.clean_only_synth  = bool(clean_only_synth)
        self.strict_target     = bool(strict_target)
        self.eps_percentile    = int(eps_percentile)
        self.eps_k_min         = int(eps_k_min)
        self.eps_k_max         = int(eps_k_max)
        self.eps_min_clip      = float(eps_min_clip)
        self.eps_max_clip      = float(eps_max_clip)
        self.min_cluster_size  = int(min_cluster_size)
        self.batch_size        = int(batch_size)
        self.max_iters         = int(max_iters)
        self.normalize_syn     = bool(normalize_syn)
        self.verbose           = int(verbose)
        self.random_state      = random_state
        self.metadata_         = {}
        self.is_fitted_        = False

    def _ts(self): return time.strftime("%H:%M:%S")

    def _log(self, msg):
        if self.verbose >= 1:
            print(f"[{self._ts()}] {msg}")

    def _rng(self):
        return check_random_state(self.random_state)

    def _to_csr(self, X):
        if issparse(X):
            return X.tocsr().astype(np.float32)
        return csr_matrix(np.asarray(X, dtype=np.float32))

    def _row_get(self, X, idx):
        if issparse(X):
            return X[idx]
        return np.asarray(X)[idx]

    def _auto_eps_min(self, Xc):
        """Heuristik eps dari k-distance percentile."""
        n     = Xc.shape[0]
        min_s = int(max(self.min_cluster_size,
                        round(np.log(max(self.min_cluster_size, n)))))
        k     = min(self.eps_k_max, max(self.eps_k_min, n - 1))

        if issparse(Xc):
            Xc_fit = Xc
        else:
            Xc_fit = np.asarray(Xc, dtype=np.float32)

        # NN untuk hitung k-distance — pakai brute karena cosine
        nn = NearestNeighbors(
            n_neighbors=k, metric=self.metric, algorithm='brute')
        nn.fit(Xc_fit)
        dist, _ = nn.kneighbors(Xc_fit)
        kth      = dist[:, k - 1]
        eps_c    = float(np.percentile(kth, self.eps_percentile))
        return float(np.clip(eps_c, self.eps_min_clip, self.eps_max_clip)), min_s

    def _smote_fill(self, X, y, targets, synth_flag):
        """Generate sampel sintetis dengan SMOTE untuk semua kelas di targets."""
        rng   = self._rng()
        X_res = X
        y_res = y
        s_res = synth_flag

        for cls, target_n in targets.items():
            idx   = np.where(y_res == cls)[0]
            cur_n = len(idx)

            if cur_n >= target_n:
                self._log(f"  ⏭ class {cls}: cur={cur_n} >= target={target_n}, skip")
                continue

            need = target_n - cur_n
            Xc   = self._row_get(X_res, idx)
            k    = max(1, min(self.k_neighbors, cur_n - 1))

            self._log(f"  🔧 class {cls}: cur={cur_n} → target={target_n} "
                      f"need={need} k={k}")

            if issparse(Xc):
                Xc_fit = Xc
            else:
                Xc_fit = np.asarray(Xc, dtype=np.float32)

            # Fit NN — kolom 0 dibuang (self-neighbor)
            nn = NearestNeighbors(
                n_neighbors=k + 1,
                metric=self.metric,
                algorithm=self.nn_algorithm)
            nn.fit(Xc_fit)
            neigh_all = nn.kneighbors(Xc_fit, return_distance=False)[:, 1:]

            synthetic_rows = []
            gen = 0
            it  = 0

            while gen < need and it < self.max_iters:
                b = min(self.batch_size, need - gen)
                for _ in range(b):
                    i     = rng.randint(0, cur_n)
                    j     = int(rng.choice(neigh_all[i]))
                    xi    = self._row_get(Xc, [i])
                    xj    = self._row_get(Xc, [j])
                    alpha = rng.rand()

                    if issparse(Xc):
                        synth = xi.multiply(1 - alpha) + xj.multiply(alpha)
                        synthetic_rows.append(synth.tocsr())
                    else:
                        synth = (np.asarray(xi) +
                                 alpha * (np.asarray(xj) - np.asarray(xi)))
                        synthetic_rows.append(synth)

                gen += b
                it  += 1

            if not synthetic_rows:
                self._log(f"  ⚠ class {cls}: gagal generate")
                continue

            if issparse(X_res):
                Xsyn  = vstack(synthetic_rows, format='csr')
                X_res = vstack([X_res, Xsyn],  format='csr')
            else:
                Xsyn  = np.vstack(synthetic_rows)
                X_res = np.vstack([X_res, Xsyn])

            ysyn  = np.full(Xsyn.shape[0], cls,  dtype=y_res.dtype)
            ssyn  = np.ones(Xsyn.shape[0],        dtype=_SYNTH_DTYPE)
            y_res = np.concatenate([y_res, ysyn])
            s_res = np.concatenate([s_res, ssyn])

            self._log(f"  ✅ class {cls}: generated={Xsyn.shape[0]} "
                      f"final={cur_n + Xsyn.shape[0]}")

        return X_res, y_res, s_res

    def _dbscan_clean(self, X, y, synth_flag, orig_counts):
        """Buang noise dengan DBSCAN. metric='cosine' → algorithm='brute'."""
        maj0      = max(orig_counts.values())
        keep_mask = np.ones(len(y), dtype=bool)
        stats     = {}

        classes_to_clean = [
            c for c in np.unique(y)
            if not (self.clean_mode == 'minor_only'
                    and orig_counts.get(int(c), 0) >= maj0)
        ]

        for c in classes_to_clean:
            idx = np.where(y == c)[0]
            if len(idx) < self.min_cluster_size:
                continue

            Xc = self._row_get(X, idx)
            if issparse(Xc):
                Xc_fit = Xc
            else:
                Xc_fit = np.asarray(Xc, dtype=np.float32)

            if self.eps == 'auto' or self.min_samples == 'auto':
                eps_c, min_s_c = self._auto_eps_min(Xc_fit)
            else:
                eps_c  = float(self.eps)
                min_s_c = int(self.min_samples)

            # ← fix: DBSCAN dengan cosine harus pakai algorithm='brute'
            db     = DBSCAN(eps=eps_c, min_samples=min_s_c,
                            metric=self.metric,
                            algorithm='brute',   # dikunci, tidak ikut nn_algorithm
                            n_jobs=-1)
            labels = db.fit_predict(Xc_fit)

            noise_local = np.where(labels == -1)[0]
            drop_global = idx[noise_local]

            if self.clean_only_synth:
                drop_global = drop_global[synth_flag[drop_global] == 1]

            n_dropped = len(drop_global)
            keep_mask[drop_global] = False

            stats[int(c)] = {
                "n_before"  : len(idx),
                "n_noise"   : len(noise_local),
                "n_dropped" : n_dropped,
                "eps_used"  : round(eps_c, 4),
                "min_s_used": min_s_c,
            }

            self._log(f"  🧹 DBSCAN class {c}: "
                      f"noise={len(noise_local)} dropped={n_dropped} "
                      f"eps={eps_c:.4f} min_s={min_s_c}")

        if issparse(X):
            X_clean = X[keep_mask]
        else:
            X_clean = X[keep_mask]

        return X_clean, y[keep_mask], synth_flag[keep_mask], stats

    def fit_resample(self, X, y, label_names=None):
        t0 = time.time()
        X  = self._to_csr(X)
        y  = np.asarray(y)

        orig_counts = dict(Counter(y))
        synth_flag  = np.zeros(X.shape[0], dtype=_SYNTH_DTYPE)
        targets     = {int(k): int(v) for k, v in self.sampling_strategy.items()}

        self._log(f"📋 Distribusi awal     : {orig_counts}")
        self._log(f"🎯 Targets             : {targets}")
        self._log(f"🔑 k_neighbors         : {self.k_neighbors}")
        self._log(f"🧹 clean_mode          : {self.clean_mode}")
        self._log(f"🧹 clean_only_synth    : {self.clean_only_synth}")
        self._log(f"🔁 strict_target       : {self.strict_target}")

        # Step 1 — SMOTE
        self._log("\n[Step 1] SMOTE...")
        X1, y1, s1 = self._smote_fill(X, y, targets, synth_flag)
        dist1 = dict(Counter(y1.tolist()))
        self._log(f"  Dist after SMOTE #1 : {dist1}")
        self._log(f"  Total sintetis      : {s1.sum():,}")

        # Step 2 — DBSCAN cleaning
        if self.clean_mode != 'none':
            self._log("\n[Step 2] DBSCAN cleaning...")
            X2, y2, s2, dbscan_stats = self._dbscan_clean(
                X1, y1, s1, orig_counts)
            dist2 = dict(Counter(y2.tolist()))
            self._log(f"  Dist after cleaning : {dist2}")
            self._log(f"  Sintetis tersisa    : {s2.sum():,}")
            self._log(f"  Sintetis dibuang    : {s1.sum() - s2.sum():,}")
        else:
            X2, y2, s2, dbscan_stats = X1, y1, s1, {}
            self._log("\n[Step 2] DBSCAN SKIPPED (clean_mode='none')")

        # Step 3 — Top-up (opsional)
        if self.strict_target:
            cnt2       = Counter(y2.tolist())
            need_again = any(cnt2.get(c, 0) < t for c, t in targets.items())
            if need_again:
                self._log("\n[Step 3] Top-up SMOTE #2...")
                X3, y3, s3 = self._smote_fill(X2, y2, targets, s2)
                dist3 = dict(Counter(y3.tolist()))
                self._log(f"  Dist after top-up   : {dist3}")
            else:
                X3, y3, s3 = X2, y2, s2
                self._log("\n[Step 3] Top-up tidak diperlukan")
        else:
            X3, y3, s3 = X2, y2, s2

        elapsed = round(time.time() - t0, 2)
        self.metadata_ = {
            "orig_counts"       : orig_counts,
            "targets"           : targets,
            "dist_after_smote1" : dist1,
            "dist_after_clean"  : dict(Counter(y2.tolist())),
            "dist_final"        : dict(Counter(y3.tolist())),
            "n_original"        : X.shape[0],
            "n_final"           : X3.shape[0],
            "total_synth_s1"    : int(s1.sum()),
            "total_synth_final" : int(s3.sum()),
            "n_dropped"         : int(s1.sum() - s2.sum()),
            "dbscan_stats"      : dbscan_stats,
            "elapsed_sec"       : elapsed,
        }
        self.is_fitted_ = True

        self._log(f"\n📦 Done: n_orig={X.shape[0]:,} "
                  f"n_final={X3.shape[0]:,} "
                  f"synth_final={s3.sum():,} "
                  f"dropped={s1.sum()-s2.sum():,} "
                  f"t={elapsed}s")

        return X3, y3

    def print_summary(self):
        if not self.is_fitted_:
            raise ValueError("Belum di-fit.")
        m = self.metadata_
        print(f"\n📊 SMOTE-RADIANT Summary")
        print(f"  Original       : {m['n_original']:,}")
        print(f"  Final          : {m['n_final']:,}")
        print(f"  Synth SMOTE #1 : {m['total_synth_s1']:,}")
        print(f"  Synth final    : {m['total_synth_final']:,}")
        print(f"  Dropped        : {m['n_dropped']:,}")
        print(f"  Drop rate      : {m['n_dropped']/(m['total_synth_s1']+1e-9)*100:.1f}%")
        print(f"  Elapsed        : {m['elapsed_sec']}s")
        if m['dbscan_stats']:
            print(f"\n  DBSCAN per kelas:")
            for c, st in m['dbscan_stats'].items():
                print(f"    class {c}: before={st['n_before']:,} "
                      f"noise={st['n_noise']:,} dropped={st['n_dropped']:,} "
                      f"eps={st['eps_used']} min_s={st['min_s_used']}")

Writing hybrid_oversampling.py
